# Agentic AI Crux

Use this starter notebook to learn agentic AI one observable step at a time. Run the setup cell once, then run each workflow step independently.

## Common setup

Reusable environment and logging setup comes from `common.py`; model creation stays explicit in the notebook.

In [ ]:
import json
import os

from openai import OpenAI

from common import configure_notebook

env_path = configure_notebook()
OPENROUTER_MODEL = "replace-with-openrouter-model-id"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

assert OPENROUTER_API_KEY, "Add OPENROUTER_API_KEY to .env first."
assert not OPENROUTER_MODEL.startswith("replace-"), "Set OPENROUTER_MODEL in this cell."
client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL)

print("STATE:", json.dumps({"env_file": env_path.name, "api_key_loaded": True, "base_url": OPENROUTER_BASE_URL, "model": OPENROUTER_MODEL}, indent=2))

## 1. Direct model call

In [ ]:
step_input = [
    {"role": "system", "content": "Answer clearly and briefly."},
    {"role": "user", "content": "What is an agentic AI workflow?"},
]
print("INPUT:", json.dumps(step_input, indent=2))
response = client.chat.completions.create(model=OPENROUTER_MODEL, messages=step_input, temperature=0)
step_output = response.choices[0]
print("OUTPUT:", step_output.message.content)
print("INTERMEDIATE RESULTS:", json.dumps({"response_id": response.id, "usage": response.usage.model_dump() if response.usage else None}, indent=2))
print("STATE:", json.dumps({"model": response.model, "finish_reason": step_output.finish_reason, "completed": True}, indent=2))

## 2. Isolated tool call

This check does not make a model or network call.

In [ ]:
def word_count(text: str) -> int:
    """Count the whitespace-separated words in text."""
    return len(text.split())

step_input = {"text": "Agent tools should be tested independently"}
print("INPUT:", json.dumps(step_input, indent=2))
step_output = word_count(**step_input)
print("OUTPUT:", step_output)
print("INTERMEDIATE RESULTS:", "Called word_count directly without a model or network request.")
print("STATE:", json.dumps({"tool": word_count.__name__, "completed": True}, indent=2))

## 3. Raw tool-calling loop

Choose an OpenRouter model that supports tool calling.

In [ ]:
tools = [{
    "type": "function",
    "function": {
        "name": "word_count",
        "description": "Count the whitespace-separated words in text.",
        "parameters": {
            "type": "object",
            "properties": {"text": {"type": "string"}},
            "required": ["text"],
            "additionalProperties": False,
        },
    },
}]
messages = [
    {"role": "system", "content": "Use tools when helpful."},
    {"role": "user", "content": "Count the words in: notebooks make API debugging clear"},
]
print("INPUT:", json.dumps({"messages": messages, "tools": tools}, indent=2))
tool_response = client.chat.completions.create(model=OPENROUTER_MODEL, messages=messages, tools=tools, tool_choice="auto", temperature=0)
assistant_message = tool_response.choices[0].message
print("OUTPUT:", assistant_message.content)
print("INTERMEDIATE RESULTS:", json.dumps([call.model_dump() for call in assistant_message.tool_calls or []], indent=2))
print("STATE:", json.dumps({"finish_reason": tool_response.choices[0].finish_reason, "tool_calls_pending": len(assistant_message.tool_calls or [])}, indent=2))

In [ ]:
assert assistant_message.tool_calls, "The selected model did not request a tool call."
assistant_payload = {
    "role": "assistant",
    "content": assistant_message.content,
    "tool_calls": [call.model_dump() for call in assistant_message.tool_calls],
}
messages.append(assistant_payload)
tool_results = []
for call in assistant_message.tool_calls:
    assert call.function.name == "word_count", f"Unknown tool requested: {call.function.name}"
    arguments = json.loads(call.function.arguments)
    result = word_count(**arguments)
    tool_result = {"role": "tool", "tool_call_id": call.id, "content": str(result)}
    messages.append(tool_result)
    tool_results.append({"name": call.function.name, "arguments": arguments, "result": result})

print("INPUT:", json.dumps([call.model_dump() for call in assistant_message.tool_calls], indent=2))
print("OUTPUT:", json.dumps(tool_results, indent=2))
print("INTERMEDIATE RESULTS:", json.dumps(messages[-len(tool_results):], indent=2))
print("STATE:", json.dumps({"message_count": len(messages), "tools_completed": len(tool_results)}, indent=2))

In [ ]:
print("INPUT:", json.dumps(messages, indent=2))
final_response = client.chat.completions.create(model=OPENROUTER_MODEL, messages=messages, tools=tools, temperature=0)
final_choice = final_response.choices[0]
print("OUTPUT:", final_choice.message.content)
print("INTERMEDIATE RESULTS:", json.dumps({"response_id": final_response.id, "usage": final_response.usage.model_dump() if final_response.usage else None}, indent=2))
print("STATE:", json.dumps({"finish_reason": final_choice.finish_reason, "completed": True}, indent=2))